# 209 — RiskAgent
Specialist agent for interpreting graph-derived risk signals.
Runs each deterministic risk task, then demonstrates `summarize_risk_for_company`
which gathers all signals and optionally produces an AI narrative.

Supported tasks: `ownership_complexity_check`, `control_signal_check`,
`address_risk_check`, `industry_context_check`, `summarize_risk_for_company`.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import uuid
import logging
import pandas as pd

from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tracing.trace_service import TraceService
from src.clients.mcp_tool_client import MCPToolClient
from src.tools.trace_tools import TraceTools
from src.agents.risk_agent import RiskAgent
from src.domain.models import InvestigationRequest, UserContext

logging.basicConfig(
    level=logging.INFO,
    format="%(name)-32s %(levelname)-8s %(message)s",
)

settings = get_neo4j_settings()
neo4j    = Neo4jRepository(**vars(settings))
neo4j.test_connection()

repo          = TraceRepository(neo4j)
trace_service = TraceService(repo)
mcp_client    = MCPToolClient()
trace_tools   = TraceTools(repo)
print("Connected")

Connected


## Companies under investigation
- **TELEFONICA O2 HOLDINGS LIMITED** — large active telco, used in notebook 208 for continuity.
- **WYELANDS BANK PLC** — financial services company with a more complex ownership profile.

In [3]:
COMPANY_A = "TELEFONICA O2 HOLDINGS LIMITED"
COMPANY_B = "GALL THOMSON ENVIRONMENTAL LIMITED"

agent = RiskAgent(mcp_client, trace_service)
print(f"agent : {agent.name}")
print(f"ai    : {agent._ai_client}")

agent : risk-agent
ai    : None


## Run deterministic risk tasks
All four tasks against a single trace. No AI client — deterministic signals only.

In [4]:
request = InvestigationRequest(
    entity_name=COMPANY_A,
    context=UserContext(
        user_id="analyst-001",
        session_id=str(uuid.uuid4()),
        metadata={"mode": "interactive"},
    ),
    request_id=str(uuid.uuid4()),
)
trace_det = trace_service.start_trace(request, request.context)
print(f"trace_id: {trace_det.request_id}")

trace_id: be37008d-0a08-415a-ae8d-e6eda14f2064


In [5]:
direct_tasks = [
    ("ownership_complexity_check", {"company_name": COMPANY_A, "max_depth": 5}),
    ("control_signal_check",       {"company_name": COMPANY_A, "max_depth": 5}),
    ("address_risk_check",         {"company_name": COMPANY_A}),
    ("industry_context_check",     {"company_name": COMPANY_A}),
]

results_det = {}
for task, ctx in direct_tasks:
    r = agent.run(task, ctx, trace_det)
    results_det[task] = r
    status = "✓" if r.success else "✗"
    print(f"[{status}] {task:<30}  {r.summary[:85]}")
    print(f"         tools: {r.tools_used}")

[✓] ownership_complexity_check      Ownership chain for 'TELEFONICA O2 HOLDINGS LIMITED': max depth 1, 1 unique owner(s),
         tools: ['ownership_complexity_check']
[✓] control_signal_check            'TELEFONICA O2 HOLDINGS LIMITED' ownership is via standard share ownership only. Cont
         tools: ['control_signal_check']
[✓] address_risk_check              'TELEFONICA O2 HOLDINGS LIMITED' shares address (BN99 3HH) with 131 other companies (
         tools: ['address_risk_check']
[✓] industry_context_check          'TELEFONICA O2 HOLDINGS LIMITED' operates under high-scrutiny SIC code(s): 74990. Pee
         tools: ['industry_context_check']


### Inspect individual results

In [6]:
r = results_det["ownership_complexity_check"]
print(f"success    : {r.success}")
print(f"summary    : {r.summary}")
if r.findings.get("ownership_complexity_check"):
    d = r.findings["ownership_complexity_check"]
    print(f"risk_level : {d['risk_level']}")
    print(f"max_depth  : {d['max_chain_depth']}")
    print(f"unique_own : {d['unique_owner_count']}")
    print(f"corp_only  : {d['corporate_chain_only']}")
    print(f"ubos       : {d['ubos']}")

success    : True
summary    : Ownership chain for 'TELEFONICA O2 HOLDINGS LIMITED': max depth 1, 1 unique owner(s), 0 individual UBO(s). Complexity risk: MEDIUM. All chains terminate at corporate entities — no individual UBOs resolved.
risk_level : MEDIUM
max_depth  : 1
unique_own : 1
corp_only  : True
ubos       : []


In [7]:
r = results_det["control_signal_check"]
print(f"success        : {r.success}")
print(f"summary        : {r.summary}")
if r.findings.get("control_signal_check"):
    d = r.findings["control_signal_check"]
    print(f"risk_level     : {d['risk_level']}")
    print(f"all_controls   : {d['all_control_types']}")
    print(f"elevated       : {d['elevated_control_types']}")
    print(f"ownership_only : {d['ownership_only']}")

success        : True
summary        : 'TELEFONICA O2 HOLDINGS LIMITED' ownership is via standard share ownership only. Control risk: LOW.
risk_level     : LOW
all_controls   : ['ownership-of-shares-75-to-100-percent']
elevated       : []
ownership_only : True


In [8]:
r = results_det["address_risk_check"]
print(f"success          : {r.success}")
print(f"summary          : {r.summary}")
if r.findings.get("address_risk_check"):
    d = r.findings["address_risk_check"]
    print(f"risk_level       : {d['risk_level']}")
    print(f"co_located_total : {d['co_located_total']}")
    print(f"co_located_active: {d['co_located_active']}")
    print(f"dissolution_rate : {d['dissolution_rate']:.0%}")

success          : True
summary          : 'TELEFONICA O2 HOLDINGS LIMITED' shares address (BN99 3HH) with 131 other companies (126 active, 5 dissolved — 4% dissolution rate). Address risk: MEDIUM.
risk_level       : MEDIUM
co_located_total : 131
co_located_active: 126
dissolution_rate : 4%


In [9]:
r = results_det["industry_context_check"]
print(f"success          : {r.success}")
print(f"summary          : {r.summary}")
if r.findings.get("industry_context_check"):
    d = r.findings["industry_context_check"]
    print(f"risk_level       : {d['risk_level']}")
    print(f"is_holding       : {d['is_holding_structure']}")
    print(f"high_scrutiny    : {d['high_scrutiny_sic_codes']}")
    print(f"peer_dissolution : {d['peer_dissolution_rate']:.0%}")

success          : True
summary          : 'TELEFONICA O2 HOLDINGS LIMITED' operates under high-scrutiny SIC code(s): 74990. Peer dissolution rate: 7% (15/200). Industry risk: MEDIUM.
risk_level       : MEDIUM
is_holding       : True
high_scrutiny    : [{'sic_code': '74990', 'reason': 'Non-trading company'}]
peer_dissolution : 7%


### Trace events — deterministic run

In [10]:
trace_service.finalize_trace(trace_det, final_summary=results_det["industry_context_check"].summary)

r = trace_tools.retrieve_trace(trace_det.request_id)
print(f"[{'✓' if r.success else '✗'}] {r.summary}")
if r.data:
    df = pd.DataFrame(r.data["events"])
    display(df[["event_number", "agent_name", "tool_name", "output_summary", "decision"]])

[✓] Trace 'be37008d-0a08-415a-ae8d-e6eda14f2064' for 'TELEFONICA O2 HOLDINGS LIMITED': 4 event(s).


,event_number,agent_name,tool_name,output_summary,decision
0,1,risk-agent,ownership_complexity_check,Ownership chain for 'TELEFONICA O2 HOLDINGS LI...,risk signal available for downstream agents
1,2,risk-agent,control_signal_check,'TELEFONICA O2 HOLDINGS LIMITED' ownership is ...,risk signal available for downstream agents
2,3,risk-agent,address_risk_check,'TELEFONICA O2 HOLDINGS LIMITED' shares addres...,risk signal available for downstream agents
3,4,risk-agent,industry_context_check,'TELEFONICA O2 HOLDINGS LIMITED' operates unde...,risk signal available for downstream agents


## summarize_risk_for_company — with AI client
Haiku (fast, cheap) vs Sonnet (richer reasoning) comparison.
Skipped automatically if `ANTHROPIC_API_KEY` is not set.

**Haiku vs Sonnet:**
- **Haiku** — default. 1-2 sentence synthesis, lowest latency and cost. Suitable for
  automated triage where speed matters.
- **Sonnet** — richer reasoning. Better at weighing multiple conflicting signals and
  producing nuanced assessments. Use for escalation review or complex cases.

In [11]:
import os

ai_client   = None
ai_settings = None
if os.getenv("ANTHROPIC_API_KEY"):
    from src.config import get_anthropic_settings
    from src.clients.anthropic_client import AnthropicClient
    ai_settings = get_anthropic_settings()
    ai_client   = AnthropicClient(settings=ai_settings)
    print(f"AI client ready")
    print(f"  haiku  : {ai_settings.model_haiku}")
    print(f"  sonnet : {ai_settings.model_sonnet}")
else:
    print("ANTHROPIC_API_KEY not set — AI cells will be skipped")

AI client ready
  haiku  : claude-haiku-4-5-20251001
  sonnet : claude-sonnet-4-6


In [12]:
trace_haiku = None
r_haiku     = None

if ai_client is None:
    print("Skipped — no AI client")
else:
    agent_ai = RiskAgent(mcp_client, trace_service, ai_client)

    req_haiku   = InvestigationRequest(
        entity_name=COMPANY_B,
        context=UserContext(
            user_id="analyst-001",
            session_id=str(uuid.uuid4()),
            metadata={"mode": "interactive", "model": "haiku"},
        ),
        request_id=str(uuid.uuid4()),
    )
    trace_haiku = trace_service.start_trace(req_haiku, req_haiku.context)
    print(f"trace_id (haiku): {trace_haiku.request_id}")

    # model=None → client default (Haiku)
    r_haiku = agent_ai.run(
        "summarize_risk_for_company",
        {"company_name": COMPANY_B},
        trace_haiku,
    )
    print(f"\n[{'✓' if r_haiku.success else '✗'}] Haiku summary:")
    print(r_haiku.summary)
    print(f"\ntools_used : {r_haiku.tools_used}")

trace_id (haiku): aa8aecd3-3f90-43a9-9b86-9d9eb6d28e0e


[03/23/26 19:24:42] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=924502;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=968569;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


[✓] Haiku summary:
GALL THOMSON ENVIRONMENTAL LIMITED presents a mixed risk profile dominated by ownership complexity concerns. The four-hop ownership chain with four distinct corporate entities and no identified individual beneficial owner triggers HIGH complexity risk, suggesting potential layering that obscures ultimate beneficial ownership — a regulatory concern under UK PSC rules. The address, control mechanisms, and industry profile are all standard and present no elevated risk, but the corporate-only chain without a named natural-person UBO requires enhanced scrutiny to establish transparency of beneficial ownership. Escalation for beneficial ownership verification is recommended. Overall risk level: HIGH.

tools_used : ['address_risk_check', 'control_signal_check', 'industry_context_check', 'ownership_complexity_check']


In [13]:
trace_sonnet = None
r_sonnet     = None

if ai_client is None:
    print("Skipped — no AI client")
else:
    req_sonnet   = InvestigationRequest(
        entity_name=COMPANY_B,
        context=UserContext(
            user_id="analyst-001",
            session_id=str(uuid.uuid4()),
            metadata={"mode": "interactive", "model": "sonnet"},
        ),
        request_id=str(uuid.uuid4()),
    )
    trace_sonnet = trace_service.start_trace(req_sonnet, req_sonnet.context)
    print(f"trace_id (sonnet): {trace_sonnet.request_id}")

    # Pass the actual Sonnet model ID via context
    r_sonnet = agent_ai.run(
        "summarize_risk_for_company",
        {"company_name": COMPANY_B, "model": ai_settings.model_sonnet},
        trace_sonnet,
    )
    print(f"\n[{'✓' if r_sonnet.success else '✗'}] Sonnet summary:")
    print(r_sonnet.summary)
    print(f"\ntools_used : {r_sonnet.tools_used}")

trace_id (sonnet): 866bf019-9a7c-46d6-b8d5-2eb2e06f93c1


[03/23/26 19:24:48] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=790294;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=899169;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


[✓] Sonnet summary:
GALL THOMSON ENVIRONMENTAL LIMITED presents a mixed risk profile driven primarily by its ownership complexity, which is rated HIGH. The ownership chain extends to a maximum depth of four hops across four distinct owner nodes, and critically, all chains terminate at corporate entities with no individual beneficial owner identified, raising material concerns under UK PSC disclosure requirements and suggesting potential layering to obscure ultimate control. This is partially offset by the remaining dimensions, which are all LOW: control mechanisms are limited to standard share ownership, the registered address at GL20 8UQ shows minimal co-location with no dissolved entities, and the company's SIC code reflects a standard industry peer group with a low dissolution rate of 10%. Nonetheless, the absence of any natural-person UBO in an otherwise moderately complex corporate chain is a standalone escalation indicator that warrants enhanced due diligence to identify and ver

In [14]:
if r_haiku and r_sonnet:
    print("=== Model comparison ===")
    print(f"\n--- Deterministic signals ---")
    print(r_haiku.findings["deterministic_summary"])
    print(f"\n--- Haiku summary ---")
    print(r_haiku.summary)
    print(f"\n--- Sonnet summary ---")
    print(r_sonnet.summary)

=== Model comparison ===

--- Deterministic signals ---
'GALL THOMSON ENVIRONMENTAL LIMITED' shares address (GL20 8UQ) with 4 other companies (4 active, 0 dissolved — 0% dissolution rate). Address risk: LOW. | 'GALL THOMSON ENVIRONMENTAL LIMITED' ownership is via standard share ownership only. Control risk: LOW. | 'GALL THOMSON ENVIRONMENTAL LIMITED' SIC codes ['32990'] are standard. Peer dissolution rate: 10% (20/200). Industry risk: LOW. | Ownership chain for 'GALL THOMSON ENVIRONMENTAL LIMITED': max depth 4, 4 unique owner(s), 0 individual UBO(s). Complexity risk: HIGH. All chains terminate at corporate entities — no individual UBOs resolved.

--- Haiku summary ---
GALL THOMSON ENVIRONMENTAL LIMITED presents a mixed risk profile dominated by ownership complexity concerns. The four-hop ownership chain with four distinct corporate entities and no identified individual beneficial owner triggers HIGH complexity risk, suggesting potential layering that obscures ultimate beneficial owners

### Trace events — Haiku run
Shows 4 `tool_returned` events (one per risk check) + 1 `agent_reasoning` event
with the AI summary and token counts in the `decision` column.

In [15]:
if trace_haiku is not None:
    trace_service.finalize_trace(trace_haiku, final_summary=r_haiku.summary)

    r = trace_tools.retrieve_trace(trace_haiku.request_id)
    print(f"[{'✓' if r.success else '✗'}] {r.summary}")
    if r.data:
        df = pd.DataFrame(r.data["events"])
        display(df[["event_number", "agent_name", "event_type", "tool_name", "decision", "why"]])

[✓] Trace 'aa8aecd3-3f90-43a9-9b86-9d9eb6d28e0e' for 'GALL THOMSON ENVIRONMENTAL LIMITED': 5 event(s).


,event_number,agent_name,event_type,tool_name,decision,why
0,1,risk-agent,tool_returned,address_risk_check,risk signal collected for synthesis,
1,2,risk-agent,tool_returned,control_signal_check,risk signal collected for synthesis,
2,3,risk-agent,tool_returned,industry_context_check,risk signal collected for synthesis,
3,4,risk-agent,tool_returned,ownership_complexity_check,risk signal collected for synthesis,
4,5,risk-agent,agent_reasoning,,AI risk summary generated for 'GALL THOMSON EN...,GALL THOMSON ENVIRONMENTAL LIMITED presents a ...


## Error cases

In [16]:
# Unknown task
r_bad = agent.run("not_a_task", {"company_name": COMPANY_A}, trace_det)
print(f"unknown task  → success={r_bad.success}  error={r_bad.error}")

# Missing company_name
r_empty = agent.run("ownership_complexity_check", {}, trace_det)
print(f"missing name  → success={r_empty.success}  error={r_empty.error}")

unknown task  → success=False  error=Unknown task 'not_a_task'. Supported tasks: address_risk_check, control_signal_check, industry_context_check, ownership_complexity_check, summarize_risk_for_company.
missing name  → success=False  error=context must include a non-empty 'company_name'.


## Cleanup — delete all traces created in this notebook

In [17]:
trace_ids = [trace_det.request_id] + (
    [trace_haiku.request_id]  if trace_haiku  else []
) + (
    [trace_sonnet.request_id] if trace_sonnet else []
)

for trace_id in trace_ids:
    neo4j.run_query(
        """
        MATCH (t:InvestigationTrace {trace_id: $trace_id})
        OPTIONAL MATCH (t)-[:HAS_EVENT]->(e:TraceEvent)
        DETACH DELETE t, e
        """,
        {"trace_id": trace_id},
    )
    print(f"Deleted {trace_id}")

Deleted be37008d-0a08-415a-ae8d-e6eda14f2064
Deleted aa8aecd3-3f90-43a9-9b86-9d9eb6d28e0e
Deleted 866bf019-9a7c-46d6-b8d5-2eb2e06f93c1


In [18]:
neo4j.close()
print("Driver closed")

Driver closed
